# Hopf CPU / GPU benchmark
Select a GPU runtime, then run these cells in order. The `colab_version` branch
must contain the updated files on GitHub before cloning. No source patching is needed.
The setup selects the assigned GPU architecture automatically.
Your original `Colab_version.ipynb` is preserved locally as a reference.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo = Path("/content/CMODE")
if not repo.exists():
    subprocess.run(["git", "clone", "--branch", "colab_version", "--single-branch",
                    "https://github.com/behzadtabari/CMODE.git", str(repo)], check=True)
branch = subprocess.check_output(["git", "-C", str(repo), "branch", "--show-current"], text=True).strip()
if branch != "colab_version":
    raise RuntimeError(f"Expected colab_version, found {branch}. Use a fresh runtime or switch the checkout.")
os.chdir(repo)
print("Repository:", repo, "Branch:", branch)


In [ ]:
# One build/install step. Run the benchmark in a fresh Python process below.
import shutil
if shutil.which("nvcc") is None:
    raise RuntimeError("CUDA compiler missing. Select a GPU runtime with the CUDA toolkit.")
subprocess.run(["nvidia-smi"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", ".[test,examples]",
                "-Ccmake.define.ODELAB_ENABLE_CUDA=ON",
                "-Ccmake.define.CMAKE_CUDA_ARCHITECTURES=native"], check=True)
subprocess.run([sys.executable, "-c",
    "import odelab; print(odelab.__file__); assert odelab.cuda_available(), 'CUDA device unavailable'"], check=True)


In [ ]:
subprocess.run([sys.executable, "examples/hopfbi_gpu_cpu_benchmarking.py",
                "--require-cuda"], check=True)


## Optional verification and plot
The benchmark checks numerical agreement before reporting each timing row.
Its CPU C++ loop calls Python for the RHS, while CUDA computes it on-device.
CUDA times include allocation, transfers, and completion, after a warm-up call.


In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "tests/python", "-q"], check=True)
subprocess.run([sys.executable, "examples/hopf_bifurcation.py",
                "--no-show", "--save", "build/hopf.png"], check=True)
from IPython.display import Image, display
display(Image(filename="build/hopf.png"))
